# Equal Earth Projection Mapping Pipeline (Standard Python Environment Edition)

This Jupyter Notebook implements the mathematical transformations for the **Equal Earth map projection** as specified by Bojan Šavrič, Tom Patterson, and Bernhard Jenny (2018). Unlike the Google Colab version, this notebook is designed to run in any standard Python environment, such as **VS Code**, a local Jupyter interface, or inside a CI/CD workflow cloned from **GitHub**.


## Background & Mathematical Breakdown

### The Problem with Mercator Maps
For centuries, the **Mercator projection** has been the standard for web maps and navigation because it preserves angles and shapes locally. However, it drastically distorts the **relative size** of landmasses as you move further from the equator. For example, on a Mercator map, Greenland appears roughly the same size as Africa, whereas Africa is actually **14 times larger** in reality.

### The Equal Earth Advantage
Developed in 2018 by Bojan Šavrič, Tom Patterson, and Bernhard Jenny, the **Equal Earth projection** is a pseudocylindrical, **equal-area projection**. It ensures that the relative sizes of all regions on the map match their actual sizes on the globe, while maintaining a visually pleasing, familiar aesthetic inspired by the Robinson projection.

### The Formula Clarified
The forward projection transforms spherical coordinates (Longitude $\lambda$ and Latitude $\varphi$) into Cartesian coordinates ($x, y$):

1. **Parametric Latitude ($\theta$):**
$$\sin \theta = \frac{\sqrt{3}}{2} \cdot \sin \varphi$$

2. **X-Coordinate (Horizontal Position):**
$$x = \frac{2 \cdot \sqrt{3} \cdot \lambda \cdot \cos \theta}{3 \cdot (A_1 + 3 \cdot A_2 \cdot \theta^2 + \theta^6 \cdot (7 \cdot A_3 + 9 \cdot A_4 \cdot \theta^2))}$$

3. **Y-Coordinate (Vertical Position):**
$$y = \theta \cdot (A_1 + A_2 \cdot \theta^2 + \theta^6 \cdot (A_3 + A_4 \cdot \theta^2))$$

> **Note on Constants:** In the provided image, $A_2$ is written as a positive value, but according to the official mathematical specification of the Equal Earth projection, $A_2$ must be negative ($A_2 = -0.081106$) to correctly adjust the curvature. The full set of official constants is:
* $A_1 = 1.340264$
* $A_2 = -0.081106$
* $A_3 = 0.000893$
* $A_4 = 0.003796$

## Step 1: Environment Setup & Dependencies

*Ensure you have installed the required packages in your local environment (`pip install geopandas shapely numpy pandas matplotlib openpyxl`).*

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon

print("Setup complete! Libraries imported successfully locally.")

## Step 2: Defining the Equal Earth Projection Functions

*This cell implements the exact formulas from the image using NumPy to support vectorized operations on entire datasets at once.*

In [ ]:
def to_equal_earth(lon_deg, lat_deg, lon_0_deg=0):
    """
    Transforms Longitude and Latitude degrees into Equal Earth X and Y coordinates.
    
    Parameters:
    - lon_deg, lat_deg: Coordinate arrays or scalars in degrees
    - lon_0_deg: Central meridian for your map (default is 0)
    """
    # Convert degrees to radians
    lam = np.radians(lon_deg - lon_0_deg)
    phi = np.radians(lat_deg)
    
    # Constants
    A1 = 1.340264
    A2 = -0.081106  # Corrected to negative based on official spec
    A3 = 0.000893
    A4 = 0.003796
    
    # 1. Calculate Parametric Latitude (theta)
    sin_theta = (np.sqrt(3) / 2.0) * np.sin(phi)
    # Clip values to avoid floating-point errors outside [-1, 1]
    sin_theta = np.clip(sin_theta, -1.0, 1.0)
    theta = np.arcsin(sin_theta)
    
    theta_sq = theta ** 2
    theta_six = theta ** 6
    cos_theta = np.cos(theta)
    
    # 2. Calculate X
    num_x = 2.0 * np.sqrt(3) * lam * cos_theta
    den_x = 3.0 * (A1 + 3.0 * A2 * theta_sq + theta_six * (7.0 * A3 + 9.0 * A4 * theta_sq))
    x = num_x / den_x
    
    # 3. Calculate Y
    y = theta * (A1 + A2 * theta_sq + theta_six * (A3 + A4 * theta_sq))
    
    return x, y

## Step 3: Unified Local Data Ingestion Pipeline

*Configure your input source here. Specify paths relative to your working directory or absolute paths on your system.*

In [ ]:
# --- CONFIGURATION BOX ---
INPUT_TYPE = "SHAPEFILE"  # Options: "CSV", "EXCEL", "SHAPEFILE"
OUTPUT_FOLDER = "./output_maps/"  # Local folder to save mappings

# File paths (Adjust these to point to your local machine paths)
CSV_PATH = "data/my_coordinates.csv"
EXCEL_PATH = "data/my_coordinates.xlsx"
# -------------------------

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def load_data():
    if INPUT_TYPE == "CSV":
        df = pd.read_csv(CSV_PATH)
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
    
    elif INPUT_TYPE == "EXCEL":
        df = pd.read_excel(EXCEL_PATH)
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
        
    elif INPUT_TYPE == "SHAPEFILE":
        # Use default dataset provided inside modern geopandas installations
        # If this path fails on older versions, download from Natural Earth directly.
        world_path = gpd.datasets.get_path('naturalearth_lowres')
        return gpd.read_file(world_path)

gdf = load_data()
print(f"Successfully loaded data. Structure contains columns: {list(gdf.columns)}")

## Step 4: Map Generation & Region Customization

*This script transforms geometries, applies geographic bounds to isolate regions (World, Europe, Africa, etc.), and sets up visualization options.*

In [ ]:
# ==============================================================================
# GEOSPATIAL FILTERING AND GEOMETRIC TRANSLATION EXPLANATIONS
# ==============================================================================
# THE INTERSECTS METHOD (.intersects()):
# By using 'gdf_input.intersects(box)', any country or polygon that is even 
# partially inside the bounding box coordinates will be captured and included 
# in its entirety. This ensures coastal islands, peninsulas, or borders 
# extending slightly beyond the boundary line are not awkwardly sliced in half 
# before being projected into the Equal Earth layout.
#
# THE CLIP METHOD (gpd.clip()):
# Unlike intersects, 'gpd.clip()' acts like a physical cookie-cutter. It 
# cleanly truncates and slices spatial geometries exactly along your bounding 
# coordinate paths. This is essential for sub-continental focus regions (such 
# as Southern Africa), where using intersects would inadvertently drag massive 
# neighboring countries (like the Democratic Republic of Congo) into the frame, 
# artificially distorting the target map ceiling.
#
# TUNING YOUR VIEWS:
# To create a tighter map crop or shift the visual canvas frame (for example, 
# adjusting the frame boundaries to include or exclude Greenland from a 
# North American layout), you can modify the coordinate vertices inside the 
# Polygon tuple arrays. The sequence must strictly follow the bounding order:
# [(Min_Longitude, Min_Latitude), 
#  (Max_Longitude, Min_Latitude), 
#  (Max_Longitude, Max_Latitude), 
#  (Min_Longitude, Max_Latitude)]
# ==============================================================================

# --- VIEWPORT CONFIGURATION ---
# Change REGION to select your map focus area. Available options:
# "WORLD", "AFRICA", "NORTHERN_AFRICA", "WESTERN_AFRICA", "EASTERN_AFRICA", 
# "SOUTHERN_AFRICA", "CENTRAL_AFRICA", "NORTH_AMERICA", "US_CANADA", 
# "CENTRAL_AMERICA", "CARIBBEAN", "SOUTH_AMERICA", "NORTHERN_SOUTH_AMERICA", 
# "SOUTHERN_CONE", "EUROPE", "ASIA", "MIDDLE_EAST", "OCEANIA", "CUSTOM"
REGION = "WORLD" 

# Custom bounding box boundaries (only used if REGION is set to "CUSTOM")
MIN_LON, MAX_LON = -20, 50
MIN_LAT, MAX_LAT = -35, 40
# ------------------------------

def filter_and_project(gdf_input, region_type):
    # Apply bounding box filters based on selection
    if region_type == "AFRICA":
        # Entire African Continent
        box = Polygon([(-25, -38), (55, -38), (55, 40), (-25, 40)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "NORTHERN_AFRICA":
        # Maghreb, Egypt, Sudan, Libya, Algeria, Morocco, Tunisia
        box = Polygon([(-20, 12), (40, 12), (40, 38), (-20, 38)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "WESTERN_AFRICA":
        # Nigeria, Ghana, Senegal, Mali, Niger, Cote d'Ivoire, etc.
        box = Polygon([(-20, 3), (25, 3), (25, 28), (-20, 28)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "EASTERN_AFRICA":
        # Horn of Africa (Somalia, Ethiopia), Kenya, Tanzania, Uganda, Madagascar
        box = Polygon([(22, -28), (52, -28), (52, 18), (22, 18)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "SOUTHERN_AFRICA":
        # South Africa, Namibia, Botswana, Zimbabwe, Mozambique
        box = Polygon([(10, -35), (40, -35), (40, -10), (10, -10)])
        # Handled via clip to preserve a sharp sub-continental boundary
        gdf_filtered = gpd.clip(gdf_input, box)
        
    elif region_type == "CENTRAL_AFRICA":
        # DRC, Angola, Congo, Gabon, Cameroon, CAR
        box = Polygon([(8, -18), (32, -18), (32, 13), (8, 13)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "NORTH_AMERICA":
        # Broad Continental Scale: Canada, USA, Greenland, Mexico
        box = Polygon([(-175, 10), (-40, 10), (-40, 85), (-175, 85)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "US_CANADA":
        # Contiguous US, southern Canada border, and Great Lakes regional focus
        box = Polygon([(-130, 22), (-60, 22), (-60, 60), (-130, 60)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "CENTRAL_AMERICA":
        # Isthmus bridge: Guatemala down to Panama
        box = Polygon([(-95, 5), (-75, 5), (-75, 20), (-95, 20)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "CARIBBEAN":
        # Caribbean Island nations, Cuba, Hispaniola, Bahamas, Lesser Antilles
        box = Polygon([(-90, 8), (-55, 8), (-55, 27), (-90, 27)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "SOUTH_AMERICA":
        # Full South American Continent frame
        box = Polygon([(-95, -60), (-30, -60), (-30, 15), (-95, 15)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "NORTHERN_SOUTH_AMERICA":
        # Amazon basin, Colombia, Venezuela, Ecuador, Peru, Guyana, Suriname
        box = Polygon([(-85, -20), (-35, -20), (-35, 14), (-85, 14)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "SOUTHERN_CONE":
        # Chile, Argentina, Uruguay, Paraguay, Southern Brazil
        box = Polygon([(-80, -56), (-45, -56), (-45, -15), (-80, -15)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "EUROPE":
        box = Polygon([(-25, 35), (45, 35), (45, 75), (-25, 75)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "ASIA":
        box = Polygon([(60, -10), (150, -10), (150, 75), (60, 75)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "MIDDLE_EAST":
        box = Polygon([(25, 12), (65, 12), (65, 43), (25, 43)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "OCEANIA":
        box = Polygon([(110, -50), (180, -50), (180, 10), (110, 10)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    elif region_type == "CUSTOM":
        box = Polygon([(MIN_LON, MIN_LAT), (MAX_LON, MIN_LAT), (MAX_LON, MAX_LAT), (MIN_LON, MAX_LAT)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
        
    else:
        gdf_filtered = gdf_input.copy()
 
    # Apply Equal Earth conversion to all spatial geometries
    projected_features = []
    for geom in gdf_filtered.geometry:
        if geom.geom_type == 'Point':
            nx, ny = to_equal_earth(geom.x, geom.y)
            projected_features.append(Point(nx, ny))
        elif geom.geom_type == 'Polygon':
            ex, ey = to_equal_earth(*geom.exterior.coords.xy)
            projected_features.append(Polygon(zip(ex, ey)))
        elif geom.geom_type == 'MultiPolygon':
            poly_list = []
            for poly in geom.geoms:
                ex, ey = to_equal_earth(*poly.exterior.coords.xy)
                poly_list.append(Polygon(zip(ex, ey)))
            projected_features.append(gpd.GeoSeries(poly_list).unary_union)
             
    gdf_projected = gpd.GeoDataFrame(gdf_filtered, geometry=projected_features)
    return gdf_projected

# Process geometries
gdf_ee = filter_and_project(gdf, REGION)
print(f"Geometries successfully projected using Equal Earth for local target view: {REGION}")

## Step 5: Render Engine and Multi-Format Exporter

*This block handles the visualization layout and exports high-quality image assets and vectorized geometry back to your computer.*

In [ ]:
# --- RENDER OPTIONS ---
OUTPUT_FILENAME = f"equal_earth_{REGION.lower()}_map"
MAP_THEME = "plasma"  # Options: 'viridis', 'plasma', 'magma', 'Blues', etc.
# ----------------------

fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
ax.set_aspect('equal')

# Plot options based on data types loaded
if 'pop_est' in gdf_ee.columns:
    # If using built-in world dataset, generate an authalic choropleth map
    gdf_ee.plot(column='pop_est', cmap=MAP_THEME, legend=True, 
                legend_kwds={'label': "Population Estimate", 'orientation': "horizontal"}, ax=ax)
else:
    # If custom user points or tracks are loaded
    gdf_ee.plot(cmap=MAP_THEME, ax=ax, markersize=15, alpha=0.8)

ax.set_title(f"Equal Earth Map Projection - Focus Region: {REGION}", fontsize=16, pad=20)
ax.axis('off') # Clean presentation style without outer boundary axes

# Define output paths
png_export_path = os.path.join(OUTPUT_FOLDER, f"{OUTPUT_FILENAME}.png")
pdf_export_path = os.path.join(OUTPUT_FOLDER, f"{OUTPUT_FILENAME}.pdf")
geojson_export_path = os.path.join(OUTPUT_FOLDER, f"{OUTPUT_FILENAME}.geojson")

# Save outputs locally
plt.savefig(png_export_path, bbox_inches='tight')
plt.savefig(pdf_export_path, bbox_inches='tight')
try:
    gdf_ee.to_file(geojson_export_path, driver="GeoJSON")
    print(f"Vector Data exported locally: {geojson_export_path}")
except Exception as e:
    print(f"GeoJSON Vector skip: Data structure requires cleaning ({e})")

plt.show()

print(f"\nSuccess! All visualizations exported safely to your output directory:")
print(f"Raster Image: {png_export_path}")
print(f"Document Presentation: {pdf_export_path}")